In [1]:
import os
import time
from tqdm import tqdm
from datetime import datetime

try:
    from api import ParsePayload
except ModuleNotFoundError:
    ! pip install catboost
    from api import ParsePayload

import json
import pandas as pd
import pickle
from functions import *
from pprint import pprint
import boto3
from passwords import *
# silence
pd.options.mode.chained_assignment = None

### Functions

In [2]:
# upload to s3
def download_from_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name, aws_session_token=None):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token,
    )
    # upload
    cls_client.download_file(
        str_project, 
        str_bucket_key, 
        str_local_path,
    )

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name, aws_session_token=None):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
try:
    str_project = os.getcwd().split('/')[4].replace('_','-')
except IndexError:
    str_project = os.getcwd().split('\\')[4].replace('_','-') 
print(f'Project: {str_project}')
str_dirname_output = './output'
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [6]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Import payload

In [7]:
str_filename = 'request_8678774_24.json'
str_local_path = f'./input/{str_filename}'
# load
try:
    dict_json_request = json.load(open(str_local_path, 'r'))['request']
except KeyError:
    dict_json_request = json.load(open(str_local_path, 'r'))

### Get lists of features in each model

In [8]:
list_cols = []
for str_model in tqdm(['01_ad','02_pricing_pd','03_pricing_lgd']):
    str_filename = 'df_cols_in_model.csv'
    str_local_path = f'./{str_filename}'
    str_bucket_key = f'{str_model}/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
    download_from_s3(
        aws_access_key_id=AWS_ACCESS_KEY_ID, 
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
        str_local_path=str_local_path, 
        str_bucket_key=str_bucket_key, 
        str_bucket_name=str_project, 
        aws_session_token=None,
    )
    time.sleep(0.5)
    list_cols_model = list(pd.read_csv(str_local_path)['feature'])
    os.remove(str_local_path)
    list_cols += list_cols_model

# rm dups
list_cols_all = list(dict.fromkeys(list_cols))
print(f'There are {len(list_cols_all)} features in {str_variant}')

100%|██████████| 3/3 [00:02<00:00,  1.48it/s]

There are 323 features in noPTImodel10


### Remove ```ENG-``` features

In [9]:
# shpw the ENG- features
list_cols_eng = [col for col in list_cols_all if 'ENG-' in col]
print(f'There are {len(list_cols_eng)} engineered features in {str_variant}:')
for a, col in enumerate(list_cols_eng):
    print(f'{a+1} - {col}')

There are 2 engineered features in noPTImodel10:
1 - ENG-vehicle_age
2 - ENG-loan_to_value


In [10]:
# rm ENG-
list_cols_raw = [col for col in list_cols_all if 'ENG-' not in col]
print(f'After removing engineered features, there are {len(list_cols_raw)} features in {str_variant}')

After removing engineered features, there are 321 features in noPTImodel10


In [11]:
# make sure the features needed for feature engineering are in the list
list_cols_force = [
    # strings
    'intopenbktype__app',
    'vehiclemake__app',
    'vehiclemodel__app',
    # date
    'applicationdate__app',    
    # ltv
    'amtfinanced__app',
    'bookvalue__app',
    # vehicle age
    'applicationdate__app',
    'vehicleyear__app',
    # dealership age
    'applicationdate__app',
    'dealerstampcreation__app',
    # other
    'fltgrossmonthly__income_sum',
    'fltgrossmonthly__income_count',
    'intterm__app',
    'intservicecontractmileage__app',
    'fltapproveddowntotal__app',
    'fltapprovedservicecontract__app',
    'fltdowncash__app',
    'fltgapinsurance__app',
    'miles_odometer__app',
    'fltadvance__app',
    # policy
    'rp01s__tu',
    'g232s__tu',
    'intopenbktype__app',
]
# merge lists
list_cols_raw = list_cols_raw + list_cols_force
# rm dups
list_cols_raw = list(dict.fromkeys(list_cols_raw))
print(f'After ensuring the features necessary for feature engineering are in the list, there are {len(list_cols_raw)} features in {str_variant}')

After ensuring the features necessary for feature engineering are in the list, there are 332 features in noPTImodel10


### Get the preprocessing script and model

In [12]:
list_str_filename = [
    #'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    str_local_path = f'./{str_filename}'
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    download_from_s3(
        aws_access_key_id=AWS_ACCESS_KEY_ID, 
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
        str_local_path=str_local_path, 
        str_bucket_key=str_bucket_path, 
        str_bucket_name=str_project, 
        aws_session_token=None,
    )
# preprocessing model
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'./{str_filename}'
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
# rm
os.remove(str_local_path)

100%|██████████| 1/1 [00:00<00:00,  4.61it/s]


### Get the inference models

In [13]:
# AD
str_filename = 'final_model.pkl'
str_bucket_path = f'01_ad/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_path, 
    str_bucket_name=str_project, 
    aws_session_token=None,
)
cls_model_inference_ad = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)

In [14]:
# PD
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_path, 
    str_bucket_name=str_project, 
    aws_session_token=None,
)
cls_model_inference_pd = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)

In [15]:
# LGD
str_filename = 'final_model.pkl'
str_bucket_path = f'03_pricing_lgd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_path, 
    str_bucket_name=str_project, 
    aws_session_token=None,
)
cls_model_inference_lgd = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)

### Import adverse action dictionary

In [16]:
# download
str_filename = 'df_aa_prod.csv'
str_local_path = f'./{str_filename}'
str_bucket_path = f'ad_hoc/make_adverse_action_dictionary/{str_variant}/{str_filename}'
download_from_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_path, 
    str_bucket_name=str_project, 
    aws_session_token=None,
)
# read
df_aa = pd.read_csv(str_local_path)
# make sure no /
df_aa['reason'] = df_aa['reason'].str.replace('/', ' or ')
# make dict
dict_aa = dict(zip(df_aa['feature'], df_aa['reason']))
# rm
os.remove(str_local_path)

#dict_aa

### Str tiers

In [17]:
str_tiers = """
{
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500,
}
"""
str_tiers = str_tiers.replace(' ','')
print(str_tiers)


{
'A1':0.0760,
'A':0.1320,
'B':0.2650,
'C':0.3220,
'D':0.3500,
}



#### Effective date - Policy

In [18]:
str_dtm_effective_date1 = '2/26/2025 00:00:00'
dtm_effective_date1 = datetime.strptime(str_dtm_effective_date1, '%m/%d/%Y %H:%M:%S')
print(f'Effective date (policy): {dtm_effective_date1}')

Effective date (policy): 2025-02-26 00:00:00


#### Effective date - Chime

In [19]:
str_dtm_effective_date2 = '3/8/2025 00:00:00'
dtm_effective_date2 = datetime.strptime(str_dtm_effective_date2, '%m/%d/%Y %H:%M:%S')
print(f'Effective date (chime): {dtm_effective_date2}')

Effective date (chime): 2025-03-08 00:00:00


### Initialize class

In [20]:
# init
cls_parse_payload = ParsePayload(
    list_cols_all=list_cols_all,
    list_cols_raw=list_cols_raw,
    cls_model_preprocessing=cls_model_preprocessing,
    cls_model_inference_ad=cls_model_inference_ad, 
    cls_model_inference_pd=cls_model_inference_pd, 
    cls_model_inference_lgd=cls_model_inference_lgd,
    dict_aa=dict_aa,
    str_tiers=str_tiers,
    dtm_effective_date1=dtm_effective_date1,
    dtm_effective_date2=dtm_effective_date2,
)

### Save locally and to s3

In [21]:
# pickle class
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
pickle.dump(cls_parse_payload, open(str_local_path, 'wb'))

# upload
str_bucket_key = f'05_parser/01_single/{str_variant}/{str_filename}'
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_key, 
    str_bucket_name=str_project, 
    aws_session_token=None,
)

# rm
del cls_parse_payload

### Load it as if it were in the API

In [22]:
%%time

# load class
cls_parse_payload = pickle.load(open(str_local_path, 'rb'))

CPU times: user 4.59 ms, sys: 70 μs, total: 4.66 ms
Wall time: 4.24 ms


### Parse payload

In [23]:
# get data
cls_parse_payload.get_data(dict_json_request)
# output
#cls_parse_payload.dict_output['dict_list_tables']

Application Date: 2025-02-24 13:06:55
Effective Date Policy: 2025-02-26 00:00:00
Effective Date Chime: 2025-03-08 00:00:00
Apply Policy: False
Apply Chime: False
[8678774106984931]: Get Data: 0.15088 sec.


In [24]:
# pmt hx fe
cls_parse_payload.engineer_pmt_hx()
# output
cls_parse_payload.dict_output['X_raw']

[8678774106984931]: Engineering payment history...


100%|██████████| 4/4 [00:00<00:00, 1937.32it/s]

Chime: True


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,intopenbktype__app,pti__app,...,int_bad_6mo_closed__tu_pmthx,int_bad_3mo_closed_end__tu_pmthx,int_bad_6mo_closed_end__tu_pmthx,list_institutions,PROGRESSRES_tag,STEP MOBILE_tag,CHIME-STRIDE_tag,ATLCAPBKSELF_tag,sum,has_inst_tag
0,8678774106984931,8678774,10698493,1,Missouri,Franchise,Missouri,True,NaN,0.1136,...,NaN,NaN,NaN,"[SELFINC/LEAD, SELFINC/LEAD, CHIME-STRIDE, SEL...",0,0,1,0,1,1


In [25]:
# shared preprocessing
cls_parse_payload.shared_preprocessing()
# output
cls_parse_payload.dict_output['X_clean']

# rm preprocessing.py
#os.remove('preprocessing.py')

NaN Replacer: 0.0020086 sec.


100%|██████████| 3/3 [00:00<00:00, 3021.83it/s]
/home/ec2-user/SageMaker/20231010_gen_xii/05_parser/01_single/preprocessing.py:98: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.replace(dict_replace, inplace=True)


Set strings: 0.0023741 sec.
Boolean Replacer: 0.0038061 sec.


100%|██████████| 326/326 [00:00<00:00, 6848.46it/s]


Data Type Setter: 0.10612 sec.


...

100%|██████████| 339/339 [00:00<00:00, 9913.75it/s]


Imputer: 0.035661 sec.
Map term: 0.00020394 sec.
Map PTI: 4.9134e-05 sec.


100%|██████████| 9/9 [00:00<00:00, 3418.65it/s]


Round values: 0.0042902 sec.
Preprocessing Model: 0.42778 sec.
[8678774106984931]: Shared Preprocessing: 0.44393 sec.


,uniqueid__app,bigaccountid__app,bigdebtorid__app,flt_wtd_avg_open__tu_pmthx,flt_wtd_avg_closed__tu_pmthx,ENG-wtd_avg,bankruptcycount24month__ln,inquiryshortterm12month__ln,re01s__tu,bankruptcystatus__ln,...,fltapprovedservicecontract__app,fltgapinsurance__app,g232s__tu,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
0,8678774106984931,8678774,10698493,NaN,NaN,NaN,0.0,0.0,2.0,0.0,...,0.0,0.0,7.0,2025,0.912673,2,1,1.0,3.0,16.89589


In [26]:
# predict
cls_parse_payload.generate_predictions()
# output
for str_key in ['y_hat_ad','mean_ad','y_hat_pd','y_hat_lgd','ecnl','ecnl_mod']:
    print('')
    print(str_key)
    print(cls_parse_payload.dict_output[str_key])

[8678774106984931]: Generate Predictions: 0.02218 sec.

y_hat_ad
0    0.555781
dtype: float64

mean_ad
0.5557810719982879

y_hat_pd
0    0.209705
dtype: float64

y_hat_lgd
0    0.637714
dtype: float64

ecnl
0.13373179596353693

ecnl_mod
0.31560703847394717


In [27]:
df_raw = cls_parse_payload.dict_output['X_raw'].copy()
list_cols = [col for col in df_raw.columns if 'list' in col]
df_raw[list_cols]

,list_pmt_hx_open__tu_pmthx,list_pmt_hx_closed__tu_pmthx,list_institutions
0,NaN,NaN,"[SELFINC/LEAD, SELFINC/LEAD, CHIME-STRIDE, SEL..."


In [28]:
# policy
cls_parse_payload.apply_policies()

No policy applied


In [29]:
# chime
cls_parse_payload.apply_chime()

No chime rules applied


In [30]:
# adverse action
cls_parse_payload.adverse_action()
# output
for list_reasons in cls_parse_payload.dict_output['list_list_reasons']:
    print('')
    pprint(list_reasons)

[8678774106984931]: Adverse Action: 0.06395 sec.

['Derogatory public record',
 'Insufficient credit file, Length of Credit',
 'Excessive Account Balances',
 'Derogatory public record',
 'Insufficient property value']


In [31]:
# counter offers
cls_parse_payload.counter_offers()
cls_parse_payload.dict_output['X_clean']

NaN Replacer: 0.0013231 sec.


100%|██████████| 3/3 [00:00<00:00, 3434.20it/s]
/home/ec2-user/SageMaker/20231010_gen_xii/05_parser/01_single/preprocessing.py:98: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.replace(dict_replace, inplace=True)


Set strings: 0.0023239 sec.
Boolean Replacer: 0.0036197 sec.


100%|██████████| 1940/1940 [00:00<00:00, 6117.76it/s]


Data Type Setter: 0.75703 sec.


...

100%|██████████| 2/2 [00:00<00:00, 1564.75it/s]


Vehicle Class: Class 3
Vehicle Class: Class 3


100%|██████████| 2/2 [00:00<00:00, 1294.54it/s]

Counter threshold: 0.33599999999999997
Initial ECNL: 0.3156
Approved (T/F): True
Tier: C
Initial offer Approved: 0.3156
Original Values:
{'LTV': 1.0086114649681528, 'APR': 0.259, 'Fees': 1554.0, 'Tier': 'C'}
Original LTV: 1.0086
Minimum LTV: 0.9078
Original APR: 0.2590
Original Fees: 1554.0000
APR + Fees Threshold: 200
There are 1 counters on amount financed
Best Counter Offer: 4
   Offer      ECNL  Decision  DownCash  AmountFinanced  SalesPrice    APR  \
0      0  0.315607  Approved      2500           19794     21799.0  0.259   
4      4  0.313007  Approved      2500           17794     19799.0  0.259   

   NetDiscount  CurrentLTV    MaxLTV  
0       1554.0    1.008611  1.008611  
4       1347.0    0.906701  1.008611  
0.259
1554.0
[8678774106984931]: Counter Offers: 3.26633 sec.


,uniqueid__app,bigaccountid__app,bigdebtorid__app,flt_wtd_avg_open__tu_pmthx,flt_wtd_avg_closed__tu_pmthx,ENG-wtd_avg,bankruptcycount24month__ln,inquiryshortterm12month__ln,re01s__tu,bankruptcystatus__ln,...,fltapprovedservicecontract__app,fltgapinsurance__app,g232s__tu,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
0,8678774106984931,8678774,10698493,NaN,NaN,NaN,0.0,0.0,2.0,0.0,...,0.0,0.0,7.0,2025,0.912673,2,1,1.0,3.0,16.89589


### Show output

In [32]:
# generate output
cls_parse_payload.generate_output()
# output
cls_parse_payload.dict_output['output_final']

1
             Row_id  Score_ad  Score_pd  Score_lgd  Score_ecnl  \
0  8678774106984931  0.555781  0.209705   0.637714    0.133732   

   Score_ecnl_mod    APR  Net_discount  \
0        0.315607  0.259        1554.0   

                                         Key_factors  Outlier_score  \
0  [Derogatory public record, Insufficient credit...            0.0   

                                          Dict_tiers  
0  \n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'...  
[{"Row_id":8678774106984931,"Score_ad":0.555781072,"Score_pd":0.2097050616,"Score_lgd":0.6377137249,"Score_ecnl":0.133731796,"Score_ecnl_mod":0.3156070385,"APR":0.259,"Net_discount":1554.0,"Key_factors":["Derogatory public record","Insufficient credit file, Length of Credit","Excessive Account Balances","Derogatory public record","Insufficient property value"],"Outlier_score":0.0,"Dict_tiers":"\n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'C':0.3220,\n'D':0.3500,\n}\n"}]
[8678774106984931]: Generate Output: 0.00523 sec.


{'Request_id': '',
 'Zaml_processing_id': '',
 'Response': [{'Model_name': 'prestige-gen-xii',
   'Model_version': 'v1',
   'Results': [{'Row_id': 8678774106984931,
     'Score_ad': 0.555781072,
     'Score_pd': 0.2097050616,
     'Score_lgd': 0.6377137249,
     'Score_ecnl': 0.133731796,
     'Score_ecnl_mod': 0.3156070385,
     'APR': 0.259,
     'Net_discount': 1554.0,
     'Key_factors': ['Derogatory public record',
      'Insufficient credit file, Length of Credit',
      'Excessive Account Balances',
      'Derogatory public record',
      'Insufficient property value'],
     'Outlier_score': 0.0,
     'Dict_tiers': "\n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'C':0.3220,\n'D':0.3500,\n}\n"}],
   'Errors': [],
   'CounterOffers': [{'Offer': 0,
     'ECNL': 0.31560703847394717,
     'Decision': 'Approved',
     'DownCash': 2500,
     'AmountFinanced': 19794,
     'SalesPrice': 21799.0,
     'APR': 0.259,
     'NetDiscount': 1554.0,
     'CurrentLTV': 1.0086114649681528,
     'Max

### Save dictionary of output locally for time analysis in the next step

In [33]:
str_filename = 'dict_output.pkl'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
pickle.dump(cls_parse_payload.dict_output, open(str_local_path, 'wb'))